# 07 — AutoTuner Validation

This notebook validates the `qc-compiler` AutoTuner implementation by exercising the search, caching, and fidelity estimation APIs with both default and real hardware models.

## 1. Setup & Imports

In [ ]:
import qc_compiler
from qc_compiler import (
    CostModel, AutoTuner, TranspileConfig, AutotuneResult,
)

print(f"qc-compiler version: {qc_compiler.__version__}")
print("All imports successful!")

## 2. Configuration Space Exploration

In [ ]:
model = CostModel()
tuner = AutoTuner(cost_model=model, cache_dir=".autotune_cache_test")

configs = tuner._generate_configurations()
print(f"Search space size: {len(configs)}")
print()

routing_methods = set(c.routing_method for c in configs)
layout_methods = set(c.layout_method for c in configs)
opt_levels = set(c.optimization_level for c in configs)
seeds = set(c.seed for c in configs)
fusion_options = set(c.gate_fusion for c in configs)
scheduling_methods = set(c.scheduling_method for c in configs)

print(f"Routing methods: {sorted(routing_methods)}")
print(f"Layout methods: {sorted(layout_methods)}")
print(f"Optimization levels: {sorted(opt_levels)}")
print(f"Seeds: {sorted(seeds)}")
print(f"Fusion options: {sorted(fusion_options)}")
print(f"Scheduling methods: {sorted(scheduling_methods)}")
print()
expected_size = len(routing_methods) * len(layout_methods) * len(opt_levels) * len(seeds) * len(fusion_options) * len(scheduling_methods)
assert len(configs) == expected_size, f"Expected {expected_size} configs, got {len(configs)}"
print(f"Verified: {len(configs)} = {len(routing_methods)} routing × {len(layout_methods)} layout × {len(opt_levels)} opt × {len(seeds)} seed × {len(fusion_options)} fusion × {len(scheduling_methods)} scheduling")

In [ ]:
default_config = TranspileConfig()
print(f"Default config key: {default_config.config_key()}")
print(f"  routing_method: {default_config.routing_method}")
print(f"  layout_method: {default_config.layout_method}")
print(f"  optimization_level: {default_config.optimization_level}")
print(f"  seed: {default_config.seed}")
print(f"  gate_fusion: {default_config.gate_fusion}")
print(f"  scheduling_method: {default_config.scheduling_method}")

config_keys = [c.config_key() for c in configs]
assert len(set(config_keys)) == len(config_keys), "Config keys must be unique"
print(f"\nAll {len(config_keys)} config keys are unique.")

## 3. Search on Bell Circuit (Default Model)

In [ ]:
from qiskit import QuantumCircuit
import shutil

bell = QuantumCircuit(2)
bell.h(0)
bell.cx(0, 1)
bell.measure_all()

cache_dir = ".autotune_cache_bell"
if shutil.os.path.exists(cache_dir):
    shutil.rmtree(cache_dir)

tuner_bell = AutoTuner(cost_model=model, cache_dir=cache_dir)
result_bell = tuner_bell.search(bell, circuit_family="bell", top_k=5)

print(f"Bell circuit search results:")
print(f"  Best config: {result_bell.best_config.config_key()}")
print(f"  Best estimated fidelity: {result_bell.best_estimated_fidelity:.6f}")
print(f"  Configs evaluated: {result_bell.circuits_evaluated}")
print(f"  Search space size: {result_bell.search_space_size}")
print(f"  Top-k configs: {len(result_bell.top_k_configs)}")
print(f"  Improvement over default: {result_bell.improvement_over_default:.6f}")

In [ ]:
assert result_bell.best_estimated_fidelity > 0, "Best fidelity should be positive"
assert result_bell.best_estimated_fidelity <= 1.0, "Best fidelity should be <= 1"
assert result_bell.circuits_evaluated > 0, "Should evaluate at least one config"
assert len(result_bell.top_k_configs) <= 5, "Top-k should not exceed requested k"
assert result_bell.best_config is not None, "Best config should not be None"
assert "default" in result_bell.all_results, "Default config should be in results"

print("All Bell circuit assertions passed!")

## 4. Search on GHZ Circuit

In [ ]:
ghz = QuantumCircuit(4)
ghz.h(0)
for i in range(1, 4):
    ghz.cx(0, i)
ghz.measure_all()

cache_dir_ghz = ".autotune_cache_ghz"
if shutil.os.path.exists(cache_dir_ghz):
    shutil.rmtree(cache_dir_ghz)

tuner_ghz = AutoTuner(cost_model=model, cache_dir=cache_dir_ghz)
result_ghz = tuner_ghz.search(ghz, circuit_family="ghz_4", top_k=5)

print(f"GHZ-4 circuit search results:")
print(f"  Best config: {result_ghz.best_config.config_key()}")
print(f"  Best estimated fidelity: {result_ghz.best_estimated_fidelity:.6f}")
print(f"  Configs evaluated: {result_ghz.circuits_evaluated}")
print(f"  Improvement over default: {result_ghz.improvement_over_default:.6f}")

In [ ]:
assert result_ghz.best_estimated_fidelity > 0, "GHZ best fidelity should be positive"
assert result_ghz.best_estimated_fidelity <= 1.0, "GHZ best fidelity should be <= 1"
assert result_ghz.best_estimated_fidelity < result_bell.best_estimated_fidelity, \
    "GHZ-4 should have lower best fidelity than Bell (more gates)"

print("All GHZ circuit assertions passed!")

## 5. Search on QAOA Circuit

In [ ]:
qaoa = QuantumCircuit(4)
for i in range(4):
    qaoa.h(i)
for i in range(3):
    qaoa.cx(i, i+1)
    qaoa.rz(0.5, i+1)
    qaoa.cx(i, i+1)
for i in range(4):
    qaoa.rx(0.3, i)
qaoa.measure_all()

cache_dir_qaoa = ".autotune_cache_qaoa"
if shutil.os.path.exists(cache_dir_qaoa):
    shutil.rmtree(cache_dir_qaoa)

tuner_qaoa = AutoTuner(cost_model=model, cache_dir=cache_dir_qaoa)
result_qaoa = tuner_qaoa.search(qaoa, circuit_family="qaoa_4", top_k=5)

print(f"QAOA-4 circuit search results:")
print(f"  Best config: {result_qaoa.best_config.config_key()}")
print(f"  Best estimated fidelity: {result_qaoa.best_estimated_fidelity:.6f}")
print(f"  Configs evaluated: {result_qaoa.circuits_evaluated}")
print(f"  Improvement over default: {result_qaoa.improvement_over_default:.6f}")

In [ ]:
assert result_qaoa.best_estimated_fidelity > 0, "QAOA best fidelity should be positive"
assert result_qaoa.best_estimated_fidelity <= 1.0, "QAOA best fidelity should be <= 1"
assert result_qaoa.best_estimated_fidelity < result_bell.best_estimated_fidelity, \
    "QAOA-4 should have lower best fidelity than Bell (more gates)"

print("All QAOA circuit assertions passed!")

## 6. Comparison of Top-k Configurations

In [ ]:
print(f"{'Rank':<5} {'Config Key':<55} {'Est. Fidelity':>13}")
print("-" * 75)
sorted_results = sorted(result_ghz.all_results.items(), key=lambda x: x[1], reverse=True)
for i, (key, fid) in enumerate(sorted_results[:10]):
    print(f"{i+1:<5} {key:<55} {fid:>13.6f}")
print(f"...")
print(f"\nTotal configs in results: {len(sorted_results)}")

In [ ]:
top_k = result_ghz.top_k_configs
print(f"\nTop-{len(top_k)} configurations for GHZ-4:")
for i, cfg in enumerate(top_k):
    key = cfg.config_key()
    fid = result_ghz.all_results.get(key, result_ghz.all_results.get("default", 0))
    print(f"  {i+1}. {key} -> fidelity={fid:.6f}")

assert len(top_k) <= 5, "Should have at most 5 top configs"
if len(top_k) >= 2:
    fid_first = result_ghz.all_results.get(top_k[0].config_key(), 0)
    fid_last = result_ghz.all_results.get(top_k[-1].config_key(), 0)
    assert fid_first >= fid_last, "Top-k should be sorted by descending fidelity"

print("\nTop-k ordering assertions passed!")

## 7. Fidelity Estimation Heuristics

In [ ]:
circuit = ghz

base_fidelity = model.estimate_fidelity(circuit).total_fidelity
print(f"Base fidelity (no config adjustments): {base_fidelity:.6f}")
print()

tuner_heuristic = AutoTuner(cost_model=model, cache_dir=".autotune_cache_heuristic")
if shutil.os.path.exists(".autotune_cache_heuristic"):
    shutil.rmtree(".autotune_cache_heuristic")

opt_levels = [0, 1, 2, 3]
print(f"{'Opt Level':>9} {'Est. Fidelity':>13} {'Delta':>8}")
print("-" * 35)
prev_fid = None
for opt in opt_levels:
    cfg = TranspileConfig(optimization_level=opt)
    fid = tuner_heuristic._estimate_fidelity(circuit, cfg)
    delta = fid - prev_fid if prev_fid is not None else 0.0
    print(f"{opt:>9} {fid:>13.6f} {delta:>8.6f}")
    prev_fid = fid

print()
fidelities_by_opt = []
for opt in opt_levels:
    cfg = TranspileConfig(optimization_level=opt)
    fidelities_by_opt.append(tuner_heuristic._estimate_fidelity(circuit, cfg))
for i in range(1, len(fidelities_by_opt)):
    assert fidelities_by_opt[i] >= fidelities_by_opt[i-1], \
        f"opt_level={opt_levels[i]} should have fidelity >= opt_level={opt_levels[i-1]}"
print("Higher optimization level => higher (or equal) fidelity: PASSED")

In [ ]:
print(f"{'Fusion':>7} {'Est. Fidelity':>13} {'Delta':>8}")
print("-" * 32)
cfg_fusion_on = TranspileConfig(gate_fusion=True)
cfg_fusion_off = TranspileConfig(gate_fusion=False)
fid_on = tuner_heuristic._estimate_fidelity(circuit, cfg_fusion_on)
fid_off = tuner_heuristic._estimate_fidelity(circuit, cfg_fusion_off)
print(f"{'True':>7} {fid_on:>13.6f}")
print(f"{'False':>7} {fid_off:>13.6f} {fid_on - fid_off:>8.6f}")

assert fid_on >= fid_off, "gate_fusion=True should have fidelity >= gate_fusion=False"
print("\nFusion ON >= Fusion OFF: PASSED")

In [ ]:
print(f"{'Scheduling':>18} {'Est. Fidelity':>13}")
print("-" * 35)
scheduling_methods = ["asap", "alap", "coherence_aware"]
sched_fidelities = {}
for sched in scheduling_methods:
    cfg = TranspileConfig(scheduling_method=sched)
    fid = tuner_heuristic._estimate_fidelity(circuit, cfg)
    sched_fidelities[sched] = fid
    print(f"{sched:>18} {fid:>13.6f}")

assert sched_fidelities["coherence_aware"] >= sched_fidelities["alap"], \
    "coherence_aware should have fidelity >= alap"
assert sched_fidelities["alap"] >= sched_fidelities["asap"], \
    "alap should have fidelity >= asap"
print("\ncoherence_aware >= alap >= asap: PASSED")

## 8. Caching Behavior

In [ ]:
cache_dir_cache = ".autotune_cache_caching"
if shutil.os.path.exists(cache_dir_cache):
    shutil.rmtree(cache_dir_cache)

tuner_cache = AutoTuner(cost_model=model, cache_dir=cache_dir_cache)

result_first = tuner_cache.search(bell, circuit_family="bell_cached", top_k=3)
print(f"First run (computes):")
print(f"  Configs evaluated: {result_first.circuits_evaluated}")
print(f"  Search space size: {result_first.search_space_size}")
print(f"  Best fidelity: {result_first.best_estimated_fidelity:.6f}")
print(f"  'cached' in all_results: {'cached' in result_first.all_results}")

assert "cached" not in result_first.all_results, "First run should compute, not use cache"
assert result_first.circuits_evaluated > 1, "First run should evaluate multiple configs"

tuner_cache2 = AutoTuner(cost_model=model, cache_dir=cache_dir_cache)
result_second = tuner_cache2.search(bell, circuit_family="bell_cached", top_k=3)
print(f"\nSecond run (from cache):")
print(f"  Configs evaluated: {result_second.circuits_evaluated}")
print(f"  Search space size: {result_second.search_space_size}")
print(f"  Best fidelity: {result_second.best_estimated_fidelity:.6f}")
print(f"  'cached' in all_results: {'cached' in result_second.all_results}")

assert "cached" in result_second.all_results, "Second run should load from cache"
assert result_second.circuits_evaluated == 1, "Second run should only evaluate cached config"
assert abs(result_second.best_estimated_fidelity - result_first.best_estimated_fidelity) < 1e-6, \
    "Cached fidelity should match original"

print("\nCaching behavior: PASSED")

In [ ]:
import json
from pathlib import Path

cache_file = Path(cache_dir_cache) / "bell_cached.json"
assert cache_file.exists(), "Cache file should exist"

with open(cache_file) as f:
    cached_data = json.load(f)

print(f"Cache file contents:")
print(f"  circuit_family: {cached_data['circuit_family']}")
print(f"  fidelity: {cached_data['fidelity']:.6f}")
print(f"  config:")
for k, v in cached_data['config'].items():
    print(f"    {k}: {v}")

assert cached_data['circuit_family'] == 'bell_cached'
assert cached_data['fidelity'] == result_first.best_estimated_fidelity
print("\nCache file structure: PASSED")

## 9. Autotuning with FakeBrisbane Backend (Real Transpilation)

In [ ]:
from qiskit_ibm_runtime.fake_provider import FakeBrisbane

backend = FakeBrisbane()
real_model = CostModel(backend=backend)

cache_dir_real = ".autotune_cache_real"
if shutil.os.path.exists(cache_dir_real):
    shutil.rmtree(cache_dir_real)

tuner_real = AutoTuner(cost_model=real_model, backend=backend, cache_dir=cache_dir_real)

print(f"Backend: {backend.name}")
print(f"Qubits: {backend.num_qubits}")

In [ ]:
bell_no_meas = QuantumCircuit(2)
bell_no_meas.h(0)
bell_no_meas.cx(0, 1)

result_real = tuner_real.search(bell_no_meas, circuit_family="bell_real", top_k=3)

print(f"FakeBrisbane autotuning results:")
print(f"  Best config: {result_real.best_config.config_key()}")
print(f"  Best estimated fidelity: {result_real.best_estimated_fidelity:.6f}")
print(f"  Configs evaluated: {result_real.circuits_evaluated}")
print(f"  Search space size: {result_real.search_space_size}")

assert 0 < result_real.best_estimated_fidelity <= 1.0, "Real backend fidelity should be in (0, 1]"
assert result_real.best_config is not None, "Should find a best config"

print("\nFakeBrisbane autotuning: PASSED")

In [ ]:
ghz_no_meas = QuantumCircuit(4)
ghz_no_meas.h(0)
for i in range(1, 4):
    ghz_no_meas.cx(0, i)

result_real_ghz = tuner_real.search(ghz_no_meas, circuit_family="ghz_real", top_k=3)

print(f"GHZ-4 on FakeBrisbane:")
print(f"  Best config: {result_real_ghz.best_config.config_key()}")
print(f"  Best estimated fidelity: {result_real_ghz.best_estimated_fidelity:.6f}")

assert 0 < result_real_ghz.best_estimated_fidelity <= 1.0
assert result_real_ghz.best_estimated_fidelity < result_real.best_estimated_fidelity, \
    "GHZ-4 should have lower fidelity than Bell on real backend"

print("\nGHZ vs Bell fidelity ordering on real backend: PASSED")

## 10. Edge Cases

In [ ]:
from dataclasses import dataclass

original_generate = AutoTuner._generate_configurations

def empty_configurations(self):
    return []

AutoTuner._generate_configurations = empty_configurations

cache_dir_empty = ".autotune_cache_empty"
if shutil.os.path.exists(cache_dir_empty):
    shutil.rmtree(cache_dir_empty)

tuner_empty = AutoTuner(cost_model=model, cache_dir=cache_dir_empty)
result_empty = tuner_empty.search(bell, circuit_family="edge_empty", top_k=3)

print(f"Empty search space result:")
print(f"  Best config: {result_empty.best_config.config_key()}")
print(f"  Best estimated fidelity: {result_empty.best_estimated_fidelity:.6f}")
print(f"  Configs evaluated: {result_empty.circuits_evaluated}")
print(f"  Search space size: {result_empty.search_space_size}")

assert result_empty.best_config is not None, "Should fall back to default config"
assert "default" in result_empty.all_results, "Default config should always be evaluated"
assert result_empty.search_space_size == 0, "Search space should be empty"

AutoTuner._generate_configurations = original_generate
print("\nEmpty search space: PASSED")

In [ ]:
def single_configuration(self):
    return [TranspileConfig(optimization_level=2, gate_fusion=True, scheduling_method="alap")]

AutoTuner._generate_configurations = single_configuration

cache_dir_single = ".autotune_cache_single"
if shutil.os.path.exists(cache_dir_single):
    shutil.rmtree(cache_dir_single)

tuner_single = AutoTuner(cost_model=model, cache_dir=cache_dir_single)
result_single = tuner_single.search(bell, circuit_family="edge_single", top_k=3)

print(f"Single-config search space result:")
print(f"  Best config: {result_single.best_config.config_key()}")
print(f"  Best estimated fidelity: {result_single.best_estimated_fidelity:.6f}")
print(f"  Configs evaluated: {result_single.circuits_evaluated}")
print(f"  Search space size: {result_single.search_space_size}")

assert result_single.best_config is not None
assert result_single.search_space_size == 1
assert result_single.circuits_evaluated >= 1

AutoTuner._generate_configurations = original_generate
print("\nSingle config search space: PASSED")

In [ ]:
cache_dir_sm = ".autotune_cache_small"
if shutil.os.path.exists(cache_dir_sm):
    shutil.rmtree(cache_dir_sm)

single_q = QuantumCircuit(1)
single_q.h(0)
single_q.measure_all()

tuner_sm = AutoTuner(cost_model=model, cache_dir=cache_dir_sm)
result_sm = tuner_sm.search(single_q, circuit_family="single_q", top_k=3)

print(f"Single-qubit circuit result:")
print(f"  Best fidelity: {result_sm.best_estimated_fidelity:.6f}")
print(f"  Configs evaluated: {result_sm.circuits_evaluated}")

assert result_sm.best_estimated_fidelity > 0.9, "Single-qubit circuit should have high fidelity"
print("\nSingle-qubit circuit: PASSED")

In [ ]:
cache_dirs = [
    ".autotune_cache_bell", ".autotune_cache_ghz", ".autotune_cache_qaoa",
    ".autotune_cache_heuristic", ".autotune_cache_caching",
    ".autotune_cache_real", ".autotune_cache_empty",
    ".autotune_cache_single", ".autotune_cache_small",
    ".autotune_cache_test",
]
for d in cache_dirs:
    if shutil.os.path.exists(d):
        shutil.rmtree(d)

print("Cleaned up all cache directories.")

## 11. Validation Summary

In [ ]:
print("=" * 60)
print("AUTOTUNER VALIDATION SUMMARY")
print("=" * 60)
print()
print("✓ Package imports (CostModel, AutoTuner, TranspileConfig, AutotuneResult)")
print("✓ Search space generation produces unique, complete configs")
print("✓ TranspileConfig.config_key() generates unique keys")
print("✓ Search on Bell circuit finds best configuration")
print("✓ Search on GHZ circuit: larger circuit has lower fidelity")
print("✓ Search on QAOA circuit: complex circuit has lower fidelity")
print("✓ Top-k configs are sorted by descending fidelity")
print("✓ Fidelity heuristics: higher opt level => higher fidelity")
print("✓ Fidelity heuristics: fusion ON >= fusion OFF")
print("✓ Fidelity heuristics: coherence_aware >= alap >= asap")
print("✓ Caching: first run computes, second run loads from cache")
print("✓ Cache file stored as JSON with correct structure")
print("✓ FakeBrisbane backend: real transpilation autotuning works")
print("✓ FakeBrisbane: GHZ has lower fidelity than Bell")
print("✓ Edge case: empty search space falls back to default")
print("✓ Edge case: single config search space works")
print("✓ Edge case: single-qubit circuit has high fidelity")
print()
print(f"Search space size: {len(tuner._generate_configurations())} configs")
print(f"Default config: {TranspileConfig().config_key()}")